# Инференс · Armenian GPT 97M

Загрузка финальной модели, 8 фиксированных продолжений, 24 вопроса и свой запрос.

**Модель:** претрейн с нуля на ArmWeb, затем 70% ArmWeb / 30% Wikipedia; без SFT.

Установи пакет из корня репозитория: %pip install -e ".[notebooks]". Путь к данным задаётся переменной LLM_STORAGE_ROOT.

Сохранённые ниже outputs получены на H100 до перестройки файлов в пакет. Вычисления и параметры генерации сохранены; обновлены импорты и пути.


In [1]:
from pathlib import Path
from datetime import datetime, timezone
from contextlib import nullcontext
from html import escape
import hashlib
import json
import os
import re
import unicodedata

import torch
import tokenizers
from tokenizers import Tokenizer
from torch.nn.attention import SDPBackend, sdpa_kernel
from IPython.display import display, HTML, Markdown

# Общие пути берутся из пакета; хранилище задаётся через LLM_STORAGE_ROOT.
os.environ.setdefault("LLM_STORAGE_ROOT", "/data/armenian_llm")
from armenian_llm.paths import STORAGE_ROOT, FACTS_CONFIG

RUN_NAME = "arm_gpt_97m_cpt_mix"
RUN_DIR = STORAGE_ROOT / "artifacts" / "checkpoints" / RUN_NAME

from armenian_llm.modeling.gpt import GPT, GPTConfig
from armenian_llm.inference.generate import generate_text

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Устройство:", torch.cuda.get_device_name(device) if device.type == "cuda" else "CPU — генерация будет медленнее")
print("Checkpoint:", RUN_DIR / "best.pt")

Устройство: NVIDIA H100 80GB HBM3
Checkpoint: /data/armenian_llm/artifacts/checkpoints/arm_gpt_97m_cpt_mix/best.pt


## Загрузка

Беру `best.pt`, выбранный по validation при обучении. Токенизатор загружается из того же запуска; SHA256 сверяется с записью в checkpoint. Оптимизатор для инференса не нужен.

У смеси шаг относится к дополнительному обучению. Общий бюджет включает токены исходной 97M. Validation печатается отдельно для каждого корпуса.

In [2]:
checkpoint_path = RUN_DIR / "best.pt"
tokenizer_path = RUN_DIR / "tokenizer" / "tokenizer.json"
checkpoint = torch.load(checkpoint_path, map_location="cpu", weights_only=True, mmap=True)

tokenizer_sha256 = hashlib.sha256(tokenizer_path.read_bytes()).hexdigest()
if tokenizer_sha256 != checkpoint["fingerprints"]["tokenizer_sha256"]:
    raise ValueError("Токенизатор отличается от использованного при обучении")

tokenizer = Tokenizer.from_file(str(tokenizer_path))
tokenizer.no_padding()
tokenizer.no_truncation()
model = GPT(GPTConfig(**checkpoint["model_config"]))
if tokenizer.get_vocab_size() != model.config.vocab_size:
    raise ValueError("Размер словаря не совпадает с конфигурацией модели")
model.load_state_dict(checkpoint["model"])
model = model.to(device).eval()

origin = checkpoint.get("origin", {})
model_info = {
    "run": RUN_NAME,
    "checkpoint": str(checkpoint_path),
    "checkpoint_step": int(checkpoint["step"]),
    "parameters": model.num_parameters(),
    "model_config": checkpoint["model_config"],
    "stage_tokens_seen": int(checkpoint["tokens_seen"]),
    "origin": origin,
    "cumulative_pretrain_tokens_seen": int(checkpoint["tokens_seen"]) + int(origin.get("tokens_seen", 0)),
    "tokenizer_sha256": tokenizer_sha256,
    "validation_losses": checkpoint.get("last_validation_losses", {}),
}
print(f"Модель: {RUN_NAME}; параметров: {model_info['parameters']:,}")
print(f"Шаг выбранного checkpoint: {model_info['checkpoint_step']:,}")
print(f"Дополнительных токенов: {model_info['stage_tokens_seen']:,}")
print(f"Всего pretrain-токенов: {model_info['cumulative_pretrain_tokens_seen']:,}")
for corpus, loss in model_info["validation_losses"].items():
    print(f"Validation {corpus}: loss {loss:.4f}")
del checkpoint

Модель: arm_gpt_97m_cpt_mix; параметров: 97,241,856
Шаг выбранного checkpoint: 5,000
Дополнительных токенов: 655,360,000
Всего pretrain-токенов: 2,992,502,784
Validation armweb: loss 2.5650
Validation wikipedia: loss 2.6329


## Отложенные тексты

Следующая ячейка показывает уже сохранённый итоговый test. Повторно оценка не запускается. Loss измеряет предсказание следующего токена, а не правильность ответа на вопрос.

In [3]:
final_test_path = STORAGE_ROOT / "artifacts" / "evaluation" / "final_97m_comparison" / "results.json"
if final_test_path.exists():
    final_test = json.loads(final_test_path.read_text(encoding="utf-8"))
    print("Источник:", final_test_path)
    print("Статус оценки:", final_test.get("status", "unknown"))
    lines = [
        "| Модель | Корпус | Test split | Loss | Perplexity |",
        "| --- | --- | --- | --- | --- |",
    ]
    for row in final_test["results"]:
        lines.append(
            f"| {row['model']} | {row['corpus']} | {row['split']} | {row['loss']:.4f} | {row['perplexity']:.2f} |"
        )
    display(Markdown("\n".join(lines)))
else:
    print("Файл с итоговой оценкой не найден. Для генерации он не требуется.")

Источник: /data/armenian_llm/artifacts/evaluation/final_97m_comparison/results.json
Статус оценки: complete


| Модель | Корпус | Test split | Loss | Perplexity |
| --- | --- | --- | --- | --- |
| arm_gpt_97m | armweb | test_iid | 2.5819 | 13.22 |
| arm_gpt_97m | armweb | test_tail | 2.6042 | 13.52 |
| arm_gpt_97m | wikipedia | test | 3.1759 | 23.95 |
| arm_gpt_97m_cpt_mix | armweb | test_iid | 2.5771 | 13.16 |
| arm_gpt_97m_cpt_mix | armweb | test_tail | 2.5979 | 13.44 |
| arm_gpt_97m_cpt_mix | wikipedia | test | 2.5700 | 13.07 |

## Продолжения текста

Те же четыре начала, два seed и настройки, что в сравнении исходной 97M со смесью. Вывожу все восемь примеров. Раскрой строку примера, чтобы увидеть полный текст.

Генерация идёт по одному токену, до EOS или лимита. Temperature 0.8 и top-k 50 задают случайную выборку. Seed фиксирует генератор случайных чисел; при другом устройстве или версии библиотек результат может отличаться. KV-cache в этом генераторе нет. После заполнения контекста используется последнее окно из 1024 токенов.

In [4]:
started_at = datetime.now(timezone.utc)
session_dir = (
    STORAGE_ROOT / "artifacts" / "inference" / RUN_NAME
    / started_at.strftime("%Y%m%d_%H%M%S_%f")
)
session_dir.mkdir(parents=True, exist_ok=False)
predictions_path = session_dir / "predictions.jsonl"
metadata_path = session_dir / "metadata.json"
session_metadata = {
    "status": "running", "started_at_utc": started_at.isoformat(),
    "model": model_info, "torch_version": str(torch.__version__),
    "tokenizers_version": tokenizers.__version__, "device": str(device),
    "gpu": torch.cuda.get_device_name(device) if device.type == "cuda" else None,
    "precision": "bf16_autocast_fp32_parameters" if device.type == "cuda" else "fp32",
    "attention": "pytorch_flash_attention" if device.type == "cuda" else "pytorch_sdpa",
    "samples_saved": 0,
}

def save_metadata():
    temporary = metadata_path.with_suffix(".json.part")
    temporary.write_text(json.dumps(session_metadata, ensure_ascii=False, indent=2), encoding="utf-8")
    temporary.replace(metadata_path)


def record_generation(kind, prompt, settings, **extra):
    attention_context = sdpa_kernel(SDPBackend.FLASH_ATTENTION) if device.type == "cuda" else nullcontext()
    with attention_context:
        continuation = generate_text(model, tokenizer, prompt, device, **settings)
    row = {
        "model": RUN_NAME, "kind": kind, "prompt": prompt,
        "settings": dict(settings), "completion": continuation,
        "text": prompt + continuation, **extra,
    }
    # Сохраняю каждый пример сразу, независимо от вывода Jupyter.
    with predictions_path.open("a", encoding="utf-8") as output:
        output.write(json.dumps(row, ensure_ascii=False) + "\n")
    session_metadata["samples_saved"] += 1
    save_metadata()
    return row


def show_example(title, text):
    display(HTML(
        "<details style='margin:12px 0'><summary>" + escape(title)
        + "</summary><pre style='white-space:pre-wrap;font-family:inherit'>"
        + escape(text) + "</pre></details>"
    ))

save_metadata()
text_settings = {"max_new_tokens": 80, "temperature": 0.8, "top_k": 50}
prompts = [
    ("Столица Армении", "Հայաստանի մայրաքաղաքը"),
    ("Сегодня в Ереване", "Այսօր Երևանում"),
    ("В истории армянской литературы", "Հայ գրականության պատմության մեջ"),
    ("Сборная Армении", "Հայաստանի հավաքականը"),
]
text_rows = []
for meaning, prompt in prompts:
    for seed in (2026, 2027):
        row = record_generation("continuation", prompt, {**text_settings, "seed": seed}, prompt_ru=meaning)
        text_rows.append(row)
        show_example(f"{meaning}; seed={seed}", row["text"])
print("Продолжений сохранено:", len(text_rows))
print("Все тексты:", predictions_path)

Продолжений сохранено: 8
Все тексты: /data/armenian_llm/artifacts/inference/arm_gpt_97m_cpt_mix/20261026_140014_127899/predictions.jsonl


## Свободные ответы на вопросы

Использую прежние 8 фактов × 3 формулировки из `benchmark_facts.json`, без few-shot примеров. Здесь оцениваю только свободную генерацию. Выбор лучшего ответа из четырёх вариантов показан в отчёте отдельно.

Формат входа: `Հարց՝ ...` / `Պատասխան՝`. Это текстовая подсказка для базовой модели, а не чат-шаблон после SFT. Top-k 1 оставляет наиболее вероятный токен; остальные настройки совпадают с прежней QA-диагностикой.

Для exact match беру первую непустую строку до армянской точки, нормализую запись и сверяю с прежним списком допустимых ответов. Полная генерация сохраняется, даже если после первой строки модель повторяет вопрос. Сам exact match не оценивает качество всего продолжения.

Набор маленький и многократно просмотренный. Факты могли встречаться в претрейне; это демонстрация ограничений модели, а не независимый бенчмарк.

In [5]:
qa_path = FACTS_CONFIG
qa_bytes = qa_path.read_bytes()
qa_dataset = json.loads(qa_bytes)
qa_dataset["few_shot_examples"] = []
qa_dataset["protocol"]["shots"] = 0
qa_settings = {"max_new_tokens": 24, "temperature": 1.0, "top_k": 1, "seed": 42}
session_metadata["qa"] = {
    "source": str(qa_path), "source_sha256": hashlib.sha256(qa_bytes).hexdigest(),
    "shots": 0, "generation": qa_settings,
    "free_answer_extraction": qa_dataset["protocol"]["free_answer_extraction"],
}
# Копия фиксирует сами вопросы и допустимые ответы этого запуска.
(session_dir / "qa_dataset.json").write_text(json.dumps(qa_dataset, ensure_ascii=False, indent=2), encoding="utf-8")
save_metadata()


def normalize_answer(text):
    text = text.strip().split("\n", 1)[0].split("։", 1)[0]
    text = unicodedata.normalize("NFC", text).casefold().replace("եւ", "և")
    return re.sub(r"\s+", " ", text).strip(" \t\r.,;:!?՝՛՜՞«»\"'—–-")


qa_rows = []
for fact in qa_dataset["facts"]:
    gold = next(choice for choice in qa_dataset["candidate_pools"][fact["category"]] if choice["text"] == fact["answer"])
    accepted = {normalize_answer(alias) for alias in gold["aliases"]}
    for variant, question in enumerate(fact["questions"], start=1):
        prompt = f"Հարց՝ {question['text']}\nՊատասխան՝"
        row = record_generation(
            "qa_zero_shot", prompt, qa_settings,
            fact_id=fact["id"], variant=variant, question=question["text"],
            translation_ru=question["translation_ru"], gold=fact["answer"],
        )
        extracted = normalize_answer(row["completion"])
        correct = extracted in accepted
        qa_rows.append({**row, "extracted_answer": extracted, "free_exact_match": correct})
        show_example(
            f"{'✓' if correct else '×'} {question['translation_ru']} — формулировка {variant}",
            f"Вопрос: {question['text']}\nЭталон: {fact['answer']}\nПолная генерация:{row['completion']}",
        )

correct_answers = sum(row["free_exact_match"] for row in qa_rows)
all_three_correct = sum(
    all(row["free_exact_match"] for row in qa_rows if row["fact_id"] == fact["id"])
    for fact in qa_dataset["facts"]
)
qa_result = {
    "questions": len(qa_rows), "facts": len(qa_dataset["facts"]),
    "free_exact_correct": correct_answers, "all_paraphrases_correct": all_three_correct,
    "predictions": qa_rows,
}
(session_dir / "qa_results.json").write_text(json.dumps(qa_result, ensure_ascii=False, indent=2), encoding="utf-8")
print(f"Свободный ответ: {correct_answers}/{len(qa_rows)}")
print(f"Все три формулировки верны: {all_three_correct}/{len(qa_dataset['facts'])}")
print("Разобранные ответы:", session_dir / "qa_results.json")

Свободный ответ: 6/24
Все три формулировки верны: 0/8
Разобранные ответы: /data/armenian_llm/artifacts/inference/arm_gpt_97m_cpt_mix/20261026_140014_127899/qa_results.json


## Свой запрос

Можно изменить `prompt` и повторить эту ячейку. Для вопроса используй тот же формат `Հարց՝ ...\nՊատասխան՝`; для продолжения текста достаточно начала фразы. Все новые варианты тоже сохраняются.

In [6]:
prompt = "Այսօր Երևանում"
custom_settings = {"max_new_tokens": 80, "temperature": 0.8, "top_k": 50, "seed": 2026}

custom_row = record_generation("custom", prompt, custom_settings)
print(custom_row["text"])

Այսօր Երևանում կբացվի «Ռոք-ն-ռոլ» երաժշտական փառատոնը
- Երևանում տեղի կունենա «Ռոք-ն-ռոլ» երաժշտական փառատոնը, որը սկիզբ է առել 1929-ին և ուղղված է ոչ միայն հայկական երաժշտության տարածմանը, այլև այդ ուղղությամբ կատարված մեծ աշխատանքի գնահատմանը: Փառատոնը հնարավորություն կընձեռի ունկնդրին ներկայացնելու հայկական դասական ռոք և կիթառային ռոք ժ


## Что показываю на защите

На отложенных текстах модель после смеси лучше предсказывает Wikipedia; на ArmWeb изменение небольшое. Это видно в test-таблице выше. Генерации позволяют отдельно разобрать связность, повторы и выдуманные факты. QA показывает, насколько ответы зависят от формулировки вопроса.

После запуска сохрани ноутбук с выводами и каталог, напечатанный ниже. В `predictions.jsonl` остаются все сгенерированные тексты, в `qa_results.json` — ответы и exact match, в `metadata.json` — checkpoint, конфигурация и настройки устройства. Для загрузки самой модели нужны `best.pt` и её токенизатор: этот ноутбук их не экспортирует.

In [7]:
session_metadata["status"] = "complete"
session_metadata["finished_at_utc"] = datetime.now(timezone.utc).isoformat()
save_metadata()
print("Примеров сохранено:", session_metadata["samples_saved"])
print("Результаты инференса:", session_dir)
print("Сохрани ноутбук через File → Save Notebook, чтобы в нём остались выводы.")

Примеров сохранено: 33
Результаты инференса: /data/armenian_llm/artifacts/inference/arm_gpt_97m_cpt_mix/20261026_140014_127899
Сохрани ноутбук через File → Save Notebook, чтобы в нём остались выводы.
